# 07-null-handling

07-null-handling/01-null-handling.py
=====================================
Covers 13 NULL handling patterns using the shared HR/ops seed dataset.

Sections
---------
 1. Detecting NULLs with isNull / isNotNull
 2. NULL audit — count NULLs per column
 3. fillna — fill NULLs with a constant
 4. dropna — drop rows with NULLs (subset + thresh)
 5. coalesce() — first non-null expression
 6. when/otherwise — conditional NULL replacement
 7. NULL-safe equality (eqNullSafe / <=>)
 8. NULL in aggregations (avg/sum/count skip NULLs)
 9. NULL in GROUP BY — creates a NULL group
10. NULL in joins — inner vs left join behaviour
11. NULL in window functions — lag() boundary
12. Replace NULL ratings before ranking
13. nanvl / isnan — NaN is NOT the same as NULL

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(__file__)), '..', '00-setup'))
from spark_session import get_spark, output_path, stop_and_wait
from seed_data import load_all, register_views
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

spark = get_spark("07-null-handling")
dfs = register_views(spark)
emp  = dfs["employees"]
pr   = dfs["performance_reviews"]
lr   = dfs["leave_requests"]
po   = dfs["purchase_orders"]
ee   = dfs["emp_events"]

## SECTION 1 — Detecting NULLs with isNull / isNotNull

In [ ]:
# DATA FLAWS surfaced:
#   emp 10, 15   → salary = None
#   emp 22       → email  = None
#   emp 35       → termination_date is set (Terminated status)

print("\n=== SECTION 1: detecting NULLs ===")

print("--- employees with NULL salary (emp 10, 15) ---")
emp.filter(F.col("salary").isNull()) \
   .select("emp_id", "first_name", "salary") \
   .show()

print("--- employees with NULL email (emp 22) ---")
emp.filter(F.col("email").isNull()) \
   .select("emp_id", "first_name", "email") \
   .show()

print("--- employees with a termination_date set (emp 35) ---")
emp.filter(F.col("termination_date").isNotNull()) \
   .select("emp_id", "status", "termination_date") \
   .show()

## SECTION 2 — NULL audit: count NULLs per column

In [ ]:
# when(isNull, 1).otherwise(0) converts nullability to a 0/1 flag;
# summing that flag gives a per-column NULL count in a single pass.

print("\n=== SECTION 2: NULL audit — count NULLs per column ===")

null_counts = emp.agg(*[
    F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(f"null_{c}")
    for c in emp.columns
])
null_counts.show(truncate=False)
# Expected non-zero: null_email=1, null_salary=2, null_dept_id=?,
# null_manager_id=?, null_termination_date=40 (most employees active)

## SECTION 3 — fillna — fill NULLs with a constant

In [ ]:
# fillna accepts a dict so different columns can have different fills.
# DATA FLAW: emp 10 and 15 get salary=0.0; emp 22 gets a placeholder email.

print("\n=== SECTION 3: fillna — replace NULLs with defaults ===")
print("DATA FLAW: emp 10, 15 salary=None → 0.0; emp 22 email=None → 'no-email@unknown.com'")

emp.fillna({"salary": 0.0, "email": "no-email@unknown.com"}) \
   .select("emp_id", "first_name", "salary", "email") \
   .filter(F.col("emp_id").isin(10, 15, 22)) \
   .show()

## SECTION 4 — dropna — remove rows that contain NULLs

In [ ]:
# dropna(subset=[...]) drops rows where ANY of the listed columns is NULL.
# dropna(thresh=N) keeps only rows with at least N non-null values.

print("\n=== SECTION 4: dropna ===")

before = emp.count()
after_salary = emp.dropna(subset=["salary"]).count()
print(f"Before dropna:           {before}")
print(f"After dropna(salary):    {after_salary}  (dropped {before - after_salary})")
print("Expected: 2 dropped — emp 10 and 15 (salary=None)")

# thresh: keep rows with at least 10 non-null columns
# employees schema has 12 columns; emp 35 has termination_date set so still >= 10
after_thresh = emp.dropna(thresh=10).count()
print(f"After dropna(thresh=10): {after_thresh}  (rows with 10+ non-null columns)")

## SECTION 5 — coalesce() — first non-null in an expression list

In [ ]:
# coalesce evaluates left-to-right and returns the first non-null.
# Here: use the real email if present, otherwise synthesise one.

print("\n=== SECTION 5: coalesce — synthesise email for emp 22 ===")
print("DATA FLAW: emp 22 email=None → coalesce builds 'emp22@internal.co'")

emp.withColumn("display_email",
    F.coalesce(
        F.col("email"),
        F.concat(F.lit("emp"), F.col("emp_id").cast("string"), F.lit("@internal.co"))
    )
) \
  .select("emp_id", "email", "display_email") \
  .filter(F.col("emp_id") == 22) \
  .show(truncate=False)

## SECTION 6 — when/otherwise — conditional NULL replacement

In [ ]:
# when().when().otherwise() chains allow multi-branch logic.
# DATA FLAWS surfaced: emp 10/15 salary=None → PENDING; emp 19 salary=0.0 → ZERO/FLAW

print("\n=== SECTION 6: when/otherwise to label salary flaws ===")
print("DATA FLAW: emp 10, 15 → PENDING; emp 19 → ZERO/FLAW")

emp.withColumn("salary_display",
    F.when(F.col("salary").isNull(), F.lit("PENDING"))
     .when(F.col("salary") == 0,    F.lit("ZERO/FLAW"))
     .otherwise(F.col("salary").cast("string"))
) \
  .select("emp_id", "salary", "salary_display") \
  .filter(F.col("emp_id").isin(10, 15, 19)) \
  .show()

## SECTION 7 — NULL-safe equality (eqNullSafe)

In [ ]:
# Regular == with NULL returns NULL (unknown), not False.
# eqNullSafe (<=>) treats NULL == NULL as True.

print("\n=== SECTION 7: NULL-safe equality ===")

count_zero = emp.filter(F.col("salary") == 0.0).count()
print(f"salary == 0.0   → {count_zero} row(s)  (finds emp 19; NULLs not matched)")

count_null_safe = emp.filter(F.col("salary").eqNullSafe(None)).count()
print(f"eqNullSafe(None) → {count_null_safe} row(s)  (finds emp 10, 15 — the NULL salary rows)")

# Regular == NULL would return 0 (NULL == NULL → NULL, not True)
count_regular_none = emp.filter(F.col("salary") == None).count()
print(f"salary == None  → {count_regular_none} row(s)  (always 0 — NULL == NULL is not True in SQL)")

## SECTION 8 — NULL in aggregations

In [ ]:
# count(*) counts all rows; count(col) counts non-null values only.
# avg/sum silently ignore NULLs — this can silently skew results.
# Replacing NULL with 0 via coalesce changes the average.

print("\n=== SECTION 8: NULL in aggregations (avg/sum/count skip NULLs) ===")
print("DATA FLAW: emp 10, 15 salary=None — excluded from avg/sum; count(*) vs count(salary) differs")

emp.agg(
    F.count("*").alias("all_rows"),           # 41
    F.count("salary").alias("non_null_salary"), # 39  (excludes emp 10, 15)
    F.avg("salary").alias("avg_excl_nulls"),    # avg of 39 rows
    F.sum("salary").alias("sum_salary"),
).show()

print("--- avg with COALESCE(salary, 0) — NULLs counted as 0, lowers average ---")
emp.withColumn("salary_zero", F.coalesce(F.col("salary"), F.lit(0.0))) \
   .agg(F.avg("salary_zero").alias("avg_incl_zeroed_nulls")) \
   .show()

## SECTION 9 — NULL in GROUP BY (NULL forms its own group)

In [ ]:
# SQL standard: NULL != NULL, but Spark's GROUP BY treats all NULLs
# as a single group key — so they aggregate together.

print("\n=== SECTION 9: NULL in GROUP BY — NULL creates its own bucket ===")
print("DATA FLAW: leave_requests row 11 has leave_type=None → appears as NULL group")

lr.groupBy("leave_type").count().orderBy("leave_type").show()
# Expected: one row with leave_type=null representing request 11 (emp 34)

## SECTION 10 — NULL in joins

In [ ]:
# INNER JOIN drops rows where the join key is NULL (NULL != anything).
# LEFT JOIN preserves all left-side rows, including NULL-key rows.

print("\n=== SECTION 10: NULL in joins — inner vs left ===")
print("DATA FLAW: some purchase_orders have dept_id=None → dropped by inner join, kept by left")

dept = dfs["departments"]

inner_count = po.join(dept, "dept_id", "inner").count()
left_count  = po.join(dept, po["dept_id"] == dept["dept_id"], "left").count()

print(f"PO rows:     {po.count()}")
print(f"INNER JOIN:  {inner_count}  (NULL dept_id rows are silently dropped)")
print(f"LEFT JOIN:   {left_count}   (NULL dept_id rows preserved, dept cols are NULL)")
# Expected: 25, 24, 25 — one row with dept_id=None excluded by inner join

## SECTION 11 — NULL in window functions (lag boundary)

In [ ]:
# lag() returns NULL for the first row in each partition (no prior row).
# salary_history has exact-dup rows 15,16 for emp 5 / 2022-04-01.

print("\n=== SECTION 11: NULL produced by lag() at partition boundary ===")
print("DATA FLAW: rows 15,16 are exact dups for emp 5 / 2022-04-01 → lag_salary=previous on dup row")

sal = dfs["salary_history"]
w   = Window.partitionBy("emp_id").orderBy("effective_date")

sal.withColumn("lag_salary", F.lag("salary_after", 1).over(w)) \
   .select("emp_id", "effective_date", "salary_after", "lag_salary") \
   .filter(F.col("emp_id") == 5) \
   .show()
# First row: lag_salary=NULL (partition start boundary)
# Dup row shows lag_salary = same salary_after (delta = 0)

## SECTION 12 — Replace NULL ratings before ranking

In [ ]:
# NULL ratings in performance_reviews skew rankings when ordering.
# Replace with 0 (or a sentinel) before ranking to make NULLs explicit.

print("\n=== SECTION 12: replace NULL performance ratings before ranking ===")
print("DATA FLAW: some review rows have rating=NULL → coalesce to 0 for safe ordering")

pr.withColumn("rating_safe", F.coalesce(F.col("rating"), F.lit(0))) \
  .select("emp_id", "review_period", "rating", "rating_safe") \
  .orderBy("review_period", "rating_safe") \
  .show(20)

## SECTION 13 — nanvl / isnan — NaN is NOT the same as NULL

In [ ]:
# NaN (Not a Number) is a special float value; NULL is the absence of a value.
# isNull() does NOT detect NaN; isnan() does NOT detect NULL.
# nanvl(a, b) returns a if a is not NaN, else b — similar to coalesce for NaN.

print("\n=== SECTION 13: NaN vs NULL — they are different! ===")
print("isNull()  → catches NULL, not NaN")
print("isnan()   → catches NaN,  not NULL")
print("nanvl()   → replaces NaN with fallback (like coalesce, but for NaN)")

nan_df = spark.createDataFrame(
    [(1, float('nan')), (2, 5.0), (3, None)],
    ["id", "val"]
)
nan_df.withColumn("has_nan",  F.isnan("val")) \
      .withColumn("is_null",  F.col("val").isNull()) \
      .withColumn("filled",   F.nanvl(F.col("val"), F.lit(0.0))) \
      .show()
# id=1: has_nan=True,  is_null=False, filled=0.0   ← NaN replaced
# id=2: has_nan=False, is_null=False, filled=5.0   ← normal
# id=3: has_nan=False, is_null=True,  filled=null  ← nanvl does NOT fix NULL

## stop_and_wait(spark)